# **MODEL EVALUATION**

In [1]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **ACTIVE LEARNING EVALUATION**

In [6]:
import json
import pandas as pd
from pathlib import Path

# Define cycle folders
cycles = {
    'Cycle 1': '/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/training01',
    'Cycle 2': '/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/training02',
    'Cycle 3': '/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/training03'
}

def load_stage1_results(results_path):
    """Load and filter Stage 1 results from results_binary.json"""
    with open(results_path, 'r') as f:
        results = json.load(f)

    # Convert list of dicts to DataFrame
    df = pd.DataFrame(results)

    # Filter to Stage 1 only
    df_s1 = df[df['experiment'].str.startswith('s1_')].copy()

    # Extract key metrics
    df_s1 = df_s1[['experiment', 'f1', 'precision', 'recall', 'balanced_accuracy', 'loss']]

    # Sort by F1 descending
    df_s1 = df_s1.sort_values('f1', ascending=False).reset_index(drop=True)

    return df_s1

## **OVERVIEW**

In [7]:
# Process each cycle
for cycle_name, folder_path in cycles.items():
    print("=" * 80)
    print(f"{cycle_name.upper()}")
    print("=" * 80)

    results_path = Path(folder_path) / 'results_binary.json'

    if not results_path.exists():
        print(f"File not found: {results_path}\n")
        continue

    df = load_stage1_results(results_path)

    print(f"\nTotal Stage 1 configs: {len(df)}")
    print(f"\nAll configs ranked by F1:\n")

    # Print all configs
    pd.set_option('display.max_rows', None)
    pd.set_option('display.float_format', '{:.3f}'.format)

    print(df.to_string(index=False))


    print("\n")

CYCLE 1

Total Stage 1 configs: 18

All configs ranked by F1:

         experiment    f1  precision  recall  balanced_accuracy  loss
s1_lr2e5_bs16_wd000 0.556      0.714   0.455              0.722 0.301
s1_lr5e5_bs08_wd001 0.533      0.421   0.727              0.835 0.238
s1_lr2e5_bs16_wd010 0.500      0.462   0.545              0.754 0.337
s1_lr3e5_bs16_wd010 0.471      0.667   0.364              0.677 0.396
s1_lr5e5_bs08_wd010 0.467      0.368   0.636              0.787 0.335
s1_lr3e5_bs16_wd001 0.467      0.368   0.636              0.787 0.209
s1_lr2e5_bs08_wd010 0.457      0.333   0.727              0.822 0.312
s1_lr3e5_bs16_wd000 0.455      0.455   0.455              0.712 0.179
s1_lr3e5_bs08_wd010 0.421      0.500   0.364              0.671 0.497
s1_lr3e5_bs08_wd001 0.409      0.273   0.818              0.846 0.473
s1_lr5e5_bs08_wd000 0.392      0.250   0.909              0.876 0.336
s1_lr5e5_bs16_wd001 0.385      0.333   0.455              0.701 0.408
s1_lr3e5_bs08_wd000 0.381  

## **F1 PATTERNS**

In [8]:
# F1 SCORES
def compare_f1_across_cycles():
    """Compare F1 scores for all configs across all cycles"""

    all_results = {}

    # Load F1 scores from each cycle
    for cycle_name, folder_path in cycles.items():
        results_path = Path(folder_path) / 'results_binary.json'

        with open(results_path, 'r') as f:
            results = json.load(f)

        df = pd.DataFrame(results)
        df_s1 = df[df['experiment'].str.startswith('s1_')].copy()

        # Store F1 scores by config name
        for _, row in df_s1.iterrows():
            config = row['experiment']
            if config not in all_results:
                all_results[config] = {}
            all_results[config][cycle_name] = row['f1']

    # Convert to DataFrame
    comparison_df = pd.DataFrame(all_results).T

    # Add delta columns
    comparison_df['Delta 1→2'] = comparison_df['Cycle 2'] - comparison_df['Cycle 1']
    comparison_df['Delta 2→3'] = comparison_df['Cycle 3'] - comparison_df['Cycle 2']
    comparison_df['Total Change'] = comparison_df['Cycle 3'] - comparison_df['Cycle 1']

    # Sort by total change (descending)
    comparison_df = comparison_df.sort_values('Total Change', ascending=False)

    return comparison_df

# Run comparison
print("=" * 100)
print("F1 COMPARISON ACROSS CYCLES")
print("=" * 100)
print("\nShowing how F1 changes for each config as data improves\n")

f1_comparison = compare_f1_across_cycles()

pd.set_option('display.float_format', '{:.3f}'.format)
print(f1_comparison.to_string())

F1 COMPARISON ACROSS CYCLES

Showing how F1 changes for each config as data improves

                     Cycle 1  Cycle 2  Cycle 3  Delta 1→2  Delta 2→3  Total Change
s1_lr5e5_bs16_wd000    0.222    0.538    0.500      0.316     -0.038         0.278
s1_lr2e5_bs16_wd001    0.308    0.600    0.545      0.292     -0.055         0.238
s1_lr2e5_bs08_wd000    0.343    0.560    0.541      0.217     -0.019         0.198
s1_lr5e5_bs08_wd000    0.392    0.457    0.583      0.065      0.126         0.191
s1_lr5e5_bs16_wd001    0.385    0.500    0.529      0.115      0.029         0.145
s1_lr2e5_bs08_wd001    0.353    0.500    0.483      0.147     -0.017         0.130
s1_lr5e5_bs16_wd010    0.296    0.545    0.421      0.249     -0.124         0.125
s1_lr3e5_bs08_wd000    0.381    0.500    0.500      0.119      0.000         0.119
s1_lr5e5_bs08_wd010    0.467    0.480    0.583      0.013      0.103         0.117
s1_lr3e5_bs08_wd010    0.421    0.571    0.476      0.150     -0.095         0.055
s

## **RECALL PATTERNS**

In [9]:
# RECALL PATTERNS
def compare_recall_across_cycles():
    """Compare recall scores for all configs across all cycles"""

    all_results = {}

    for cycle_name, folder_path in cycles.items():
        results_path = Path(folder_path) / 'results_binary.json'

        with open(results_path, 'r') as f:
            results = json.load(f)

        df = pd.DataFrame(results)
        df_s1 = df[df['experiment'].str.startswith('s1_')].copy()

        for _, row in df_s1.iterrows():
            config = row['experiment']
            if config not in all_results:
                all_results[config] = {}
            all_results[config][cycle_name] = row['recall']

    comparison_df = pd.DataFrame(all_results).T
    comparison_df['Delta 1→2'] = comparison_df['Cycle 2'] - comparison_df['Cycle 1']
    comparison_df['Delta 2→3'] = comparison_df['Cycle 3'] - comparison_df['Cycle 2']
    comparison_df['Total Change'] = comparison_df['Cycle 3'] - comparison_df['Cycle 1']

    comparison_df = comparison_df.sort_values('Total Change', ascending=False)

    return comparison_df

recall_comparison = compare_recall_across_cycles()

pd.set_option('display.float_format', '{:.3f}'.format)
print(recall_comparison.to_string())

                     Cycle 1  Cycle 2  Cycle 3  Delta 1→2  Delta 2→3  Total Change
s1_lr5e5_bs16_wd001    0.455    0.545    0.818      0.091      0.273         0.364
s1_lr3e5_bs16_wd010    0.364    0.727    0.727      0.364     -0.000         0.364
s1_lr2e5_bs08_wd001    0.273    0.455    0.636      0.182      0.182         0.364
s1_lr2e5_bs16_wd001    0.182    0.545    0.545      0.364     -0.000         0.364
s1_lr2e5_bs08_wd000    0.545    0.636    0.909      0.091      0.273         0.364
s1_lr3e5_bs08_wd000    0.364    0.545    0.636      0.182      0.091         0.273
s1_lr5e5_bs16_wd000    0.273    0.636    0.545      0.364     -0.091         0.273
s1_lr3e5_bs16_wd000    0.455    0.455    0.636      0.000      0.182         0.182
s1_lr2e5_bs16_wd010    0.545    0.455    0.727     -0.091      0.273         0.182
s1_lr3e5_bs08_wd010    0.364    0.727    0.455      0.364     -0.273         0.091
s1_lr5e5_bs16_wd010    0.364    0.545    0.364      0.182     -0.182         0.000
s1_l

## **PRECISION PATTERNS**

In [10]:
def compare_precision_across_cycles():
    """Compare precision scores for all configs across all cycles"""

    all_results = {}

    for cycle_name, folder_path in cycles.items():
        results_path = Path(folder_path) / 'results_binary.json'

        with open(results_path, 'r') as f:
            results = json.load(f)

        df = pd.DataFrame(results)
        df_s1 = df[df['experiment'].str.startswith('s1_')].copy()

        for _, row in df_s1.iterrows():
            config = row['experiment']
            if config not in all_results:
                all_results[config] = {}
            all_results[config][cycle_name] = row['precision']

    comparison_df = pd.DataFrame(all_results).T
    comparison_df['Delta 1→2'] = comparison_df['Cycle 2'] - comparison_df['Cycle 1']
    comparison_df['Delta 2→3'] = comparison_df['Cycle 3'] - comparison_df['Cycle 2']
    comparison_df['Total Change'] = comparison_df['Cycle 3'] - comparison_df['Cycle 1']

    comparison_df = comparison_df.sort_values('Total Change', ascending=False)

    return comparison_df

precision_comparison = compare_precision_across_cycles()

pd.set_option('display.float_format', '{:.3f}'.format)
print(precision_comparison.to_string())

                     Cycle 1  Cycle 2  Cycle 3  Delta 1→2  Delta 2→3  Total Change
s1_lr5e5_bs08_wd000    0.250    0.333    0.538      0.083      0.205         0.288
s1_lr5e5_bs16_wd000    0.188    0.467    0.462      0.279     -0.005         0.274
s1_lr5e5_bs16_wd010    0.250    0.545    0.500      0.295     -0.045         0.250
s1_lr5e5_bs08_wd010    0.368    0.429    0.538      0.060      0.110         0.170
s1_lr3e5_bs08_wd001    0.273    0.455    0.417      0.182     -0.038         0.144
s1_lr2e5_bs08_wd000    0.250    0.500    0.385      0.250     -0.115         0.135
s1_lr2e5_bs08_wd010    0.333    0.467    0.412      0.133     -0.055         0.078
s1_lr5e5_bs16_wd001    0.333    0.462    0.391      0.128     -0.070         0.058
s1_lr3e5_bs08_wd000    0.400    0.462    0.412      0.061     -0.050         0.012
s1_lr3e5_bs08_wd010    0.500    0.471    0.500     -0.029      0.029         0.000
s1_lr5e5_bs08_wd001    0.421    0.276    0.400     -0.145      0.124        -0.021
s1_l

## **TRAINING STABILITY PATTERNS - FULL VIEW**

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

def analyze_training_dynamics_from_logs():
    """Analyze training stability from epoch-by-epoch logs"""

    results = []

    for cycle_name, folder in cycles.items():
        # Get all Stage 1 configs
        results_path = Path(folder) / 'results_binary.json'
        with open(results_path, 'r') as f:
            cycle_results = json.load(f)

        df = pd.DataFrame(cycle_results)
        configs = df[df['experiment'].str.startswith('s1_')]['experiment'].tolist()

        for config in configs:
            # Load training logs
            log_path = Path(folder) / config / 'training_logs.csv'
            if not log_path.exists():
                continue

            logs = pd.read_csv(log_path)

            # Separate training and eval rows
            train_rows = logs[logs['loss'].notna()].copy()
            eval_rows = logs[logs['eval_loss'].notna()].copy()

            # Group by epoch and take last value
            train_by_epoch = train_rows.groupby('epoch')['loss'].last()
            eval_by_epoch = eval_rows.groupby('epoch').agg({
                'eval_loss': 'last',
                'eval_f1': 'last',
                'eval_precision': 'last',
                'eval_recall': 'last'
            })

            # Merge on epoch
            merged = pd.merge(
                train_by_epoch,
                eval_by_epoch,
                left_index=True,
                right_index=True,
                how='inner'
            )

            if len(merged) == 0:
                continue

            # Calculate overfitting gap
            merged['overfit_gap'] = merged['eval_loss'] - merged['loss']

            # Final epoch values
            final_train_loss = merged['loss'].iloc[-1]
            final_eval_loss = merged['eval_loss'].iloc[-1]
            final_gap = merged['overfit_gap'].iloc[-1]

            # Average gap across all epochs
            mean_gap = merged['overfit_gap'].mean()

            # Gap variability
            gap_std = merged['overfit_gap'].std()

            # Eval loss variability
            eval_loss_std = merged['eval_loss'].std()

            # Check convergence (last 5 epochs)
            if len(merged) >= 5:
                last_5_eval_std = merged['eval_loss'].tail(5).std()
                converged = last_5_eval_std < 0.05
            else:
                last_5_eval_std = np.nan
                converged = False

            results.append({
                'cycle': cycle_name,
                'config': config,
                'final_train_loss': final_train_loss,
                'final_eval_loss': final_eval_loss,
                'final_gap': final_gap,
                'mean_gap': mean_gap,
                'gap_std': gap_std,
                'eval_loss_std': eval_loss_std,
                'last_5_eval_std': last_5_eval_std,
                'converged': converged,
                'num_epochs': len(merged)
            })

    return pd.DataFrame(results)

### **OVERFITTING MEAN GAP**

In [12]:
# Run analysis
dynamics_df = analyze_training_dynamics_from_logs()

print("\n" + "=" * 90)
print("OVERFITTING GAP ANALYSIS")
print("=" * 90)
print("\nGap = eval_loss - train_loss")

# Pivot: Mean gap per config per cycle
pivot_mean = dynamics_df.pivot_table(
    index='config',
    columns='cycle',
    values='mean_gap',
    aggfunc='first'
)

# Add delta columns
pivot_mean['Delta 1→2'] = pivot_mean['Cycle 2'] - pivot_mean['Cycle 1']
pivot_mean['Delta 2→3'] = pivot_mean['Cycle 3'] - pivot_mean['Cycle 2']
pivot_mean['Total Change'] = pivot_mean['Cycle 3'] - pivot_mean['Cycle 1']

# Reorder columns
pivot_mean = pivot_mean[['Cycle 1', 'Cycle 2', 'Cycle 3', 'Delta 1→2', 'Delta 2→3', 'Total Change']]

# Sort by total change
pivot_mean = pivot_mean.sort_values('Total Change')

pd.set_option('display.float_format', '{:.3f}'.format)
print(pivot_mean.to_string())


OVERFITTING GAP ANALYSIS

Gap = eval_loss - train_loss
cycle                Cycle 1  Cycle 2  Cycle 3  Delta 1→2  Delta 2→3  Total Change
config                                                                            
s1_lr2e5_bs08_wd001    0.439    0.436    0.334     -0.003     -0.103        -0.105
s1_lr2e5_bs16_wd001    0.387    0.301    0.330     -0.086      0.029        -0.057
s1_lr5e5_bs16_wd000    0.322    0.347    0.270      0.025     -0.077        -0.052
s1_lr5e5_bs08_wd000    0.376    0.332    0.325     -0.045     -0.007        -0.052
s1_lr2e5_bs16_wd010    0.371    0.312    0.358     -0.060      0.047        -0.013
s1_lr3e5_bs16_wd010    0.400    0.420    0.390      0.020     -0.030        -0.010
s1_lr2e5_bs16_wd000    0.323    0.288    0.330     -0.035      0.042         0.007
s1_lr3e5_bs16_wd000    0.309    0.261    0.317     -0.048      0.056         0.008
s1_lr3e5_bs08_wd010    0.458    0.336    0.492     -0.122      0.156         0.034
s1_lr3e5_bs08_wd001    0.383   

### **GAP STABILITY**

In [ ]:
print("\n" + "=" * 90)
print("OVERFITTING GAP STABILITY")
print("=" * 90)
print("\nLower std = more stable training")
print("Negative change = more stable\n")

pivot_stability = dynamics_df.pivot_table(
    index='config',
    columns='cycle',
    values='gap_std',
    aggfunc='first'
)

# Add delta columns
pivot_stability['Delta 1→2'] = pivot_stability['Cycle 2'] - pivot_stability['Cycle 1']
pivot_stability['Delta 2→3'] = pivot_stability['Cycle 3'] - pivot_stability['Cycle 2']
pivot_stability['Total Change'] = pivot_stability['Cycle 3'] - pivot_stability['Cycle 1']

# Reorder columns
pivot_stability = pivot_stability[['Cycle 1', 'Cycle 2', 'Cycle 3', 'Delta 1→2', 'Delta 2→3', 'Total Change']]

# Sort by total change
pivot_stability = pivot_stability.sort_values('Total Change')

print(pivot_stability.to_string())


OVERFITTING GAP STABILITY

Lower std = more stable training
Negative change = more stable

cycle                Cycle 1  Cycle 2  Cycle 3  Delta 1→2  Delta 2→3  Total Change
config                                                                            
s1_lr3e5_bs16_wd001    0.298    0.272    0.185     -0.026     -0.087        -0.113
s1_lr2e5_bs16_wd000    0.276    0.292    0.179      0.016     -0.113        -0.097
s1_lr5e5_bs08_wd001    0.297    0.229    0.213     -0.068     -0.016        -0.084
s1_lr2e5_bs16_wd010    0.280    0.297    0.199      0.017     -0.098        -0.081
s1_lr2e5_bs08_wd000    0.282    0.246    0.204     -0.036     -0.042        -0.078
s1_lr2e5_bs16_wd001    0.283    0.309    0.213      0.026     -0.097        -0.070
s1_lr5e5_bs16_wd001    0.218    0.266    0.162      0.047     -0.103        -0.056
s1_lr3e5_bs16_wd000    0.308    0.278    0.258     -0.030     -0.021        -0.051
s1_lr3e5_bs16_wd010    0.270    0.255    0.236     -0.015     -0.018        -0

## **CONVERGENCE PATTERNS**

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

def analyze_final_epochs(n_final=6):
    """Analyze last N epochs to assess convergence quality"""

    results = []

    for cycle_name, folder in cycles.items():
        # Get all Stage 1 configs
        results_path = Path(folder) / 'results_binary.json'
        with open(results_path, 'r') as f:
            cycle_results = json.load(f)

        df = pd.DataFrame(cycle_results)
        configs = df[df['experiment'].str.startswith('s1_')]['experiment'].tolist()

        for config in configs:
            # Load training logs
            log_path = Path(folder) / config / 'training_logs.csv'
            if not log_path.exists():
                continue

            logs = pd.read_csv(log_path)

            # Separate training and eval rows
            train_rows = logs[logs['loss'].notna()].copy()
            eval_rows = logs[logs['eval_loss'].notna()].copy()

            # Group by epoch and merge
            train_by_epoch = train_rows.groupby('epoch')['loss'].last()
            eval_by_epoch = eval_rows.groupby('epoch').agg({
                'eval_loss': 'last',
                'eval_f1': 'last',
                'eval_precision': 'last',
                'eval_recall': 'last'
            })

            # Merge on epoch
            merged = pd.merge(
                train_by_epoch,
                eval_by_epoch,
                left_index=True,
                right_index=True,
                how='inner'
            )

            # Calculate overfitting gap
            merged['overfit_gap'] = merged['eval_loss'] - merged['loss']

            if len(merged) < n_final:
              continue

            # Get last N epochs
            final_epochs = merged.tail(n_final)

            # F1 metrics for final epochs
            final_f1_mean = final_epochs['eval_f1'].mean()
            final_f1_trend = final_epochs['eval_f1'].iloc[-1] - final_epochs['eval_f1'].iloc[0]
            final_f1_std = final_epochs['eval_f1'].std()
            final_f1_max = final_epochs['eval_f1'].max()
            final_f1_min = final_epochs['eval_f1'].min()

            # Gap metrics for final epochs
            final_gap_mean = final_epochs['overfit_gap'].mean()
            final_gap_trend = final_epochs['overfit_gap'].iloc[-1] - final_epochs['overfit_gap'].iloc[0]
            final_gap_std = final_epochs['overfit_gap'].std()
            final_gap_max = final_epochs['overfit_gap'].max()
            final_gap_min = final_epochs['overfit_gap'].min()

            final_loss_std = final_epochs['eval_loss'].std()

            # Convergence quality indicators
            f1_converged = final_f1_std < 0.02
            gap_stable = final_gap_std < 0.01
            still_improving = final_f1_trend > 0.02
            still_overfitting = final_gap_trend > 0.02

            results.append({
                'cycle': cycle_name,
                'config': config,
                'final_f1_mean': final_f1_mean,
                'final_f1_trend': final_f1_trend,
                'final_f1_std': final_f1_std,
                'final_f1_max': final_f1_max,
                'final_f1_min': final_f1_min,
                'final_gap_mean': final_gap_mean,
                'final_gap_trend': final_gap_trend,
                'final_gap_std': final_gap_std,
                'final_gap_max': final_gap_max,
                'final_gap_min': final_gap_min,
                'final_loss_std': final_loss_std,
                'f1_converged': f1_converged,
                'gap_stable': gap_stable,
                'still_improving': still_improving,
                'still_overfitting': still_overfitting,
                'num_epochs': len(merged)
            })

    return pd.DataFrame(results)

# Run analysis
final_epoch_df = analyze_final_epochs(n_final=6)

In [24]:
def extract_table_values(final_epoch_df, cycles_dict, n_final=6):
    """Extract pooled F1 and gap values from last 6 epochs across all configs"""

    table_results = []

    for cycle_name, folder in cycles_dict.items():
        all_f1_values = []
        all_gap_values = []

        # Get all Stage 1 configs
        results_path = Path(folder) / 'results_binary.json'
        with open(results_path, 'r') as f:
            cycle_results = json.load(f)

        df = pd.DataFrame(cycle_results)
        configs = df[df['experiment'].str.startswith('s1_')]['experiment'].tolist()

        for config in configs:
            log_path = Path(folder) / config / 'training_logs.csv'
            if not log_path.exists():
                continue

            logs = pd.read_csv(log_path)

            # Separate training and eval rows
            train_rows = logs[logs['loss'].notna()].copy()
            eval_rows = logs[logs['eval_loss'].notna()].copy()

            # Group by epoch and merge
            train_by_epoch = train_rows.groupby('epoch')['loss'].last()
            eval_by_epoch = eval_rows.groupby('epoch').agg({
                'eval_loss': 'last',
                'eval_f1': 'last'
            })

            merged = pd.merge(train_by_epoch, eval_by_epoch,
                            left_index=True, right_index=True, how='inner')

            merged['overfit_gap'] = merged['eval_loss'] - merged['loss']

            if len(merged) < n_final:
                continue

            # Get last N epochs
            final_epochs = merged.tail(n_final)

            # Collect all F1 and gap values from these epochs
            all_f1_values.extend(final_epochs['eval_f1'].tolist())
            all_gap_values.extend(final_epochs['overfit_gap'].tolist())

        # Compute statistics across all pooled values
        table_results.append({
            'cycle': cycle_name,
            'f1_mean': np.mean(all_f1_values),
            'f1_max': np.max(all_f1_values),
            'f1_std': np.std(all_f1_values),
            'gap_mean': np.mean(all_gap_values),
            'gap_max': np.max(all_gap_values),
            'gap_std': np.std(all_gap_values)
        })

    return pd.DataFrame(table_results)

# Run the extraction
table_values = extract_table_values(final_epoch_df, cycles, n_final=6)

print("\nTable Values:")
print(table_values.round(3))

print("\n\nLaTeX Format:")
for _, row in table_values.iterrows():
    cycle_name = row['cycle']
    if 'Cycle 3' in cycle_name or cycle_name == 'cycle3':
        print(f"Cycle 3 & \\textbf{{{row['f1_mean']:.3f}}} & \\textbf{{{row['f1_max']:.3f}}} & \\textbf{{{row['f1_std']:.3f}}} & {row['gap_mean']:.3f} & {row['gap_max']:.3f} & {row['gap_std']:.3f} \\\\")
    else:
        cycle_num = cycle_name.split()[-1] if ' ' in cycle_name else cycle_name[-1]
        print(f"Cycle {cycle_num} & {row['f1_mean']:.3f} & {row['f1_max']:.3f} & {row['f1_std']:.3f} & {row['gap_mean']:.3f} & {row['gap_max']:.3f} & {row['gap_std']:.3f} \\\\")


Table Values:
     cycle  f1_mean  f1_max  f1_std  gap_mean  gap_max  gap_std
0  Cycle 1    0.231   0.632   0.121     0.411    0.658    0.205
1  Cycle 2    0.338   0.600   0.130     0.389    0.662    0.229
2  Cycle 3    0.391   0.583   0.093     0.492    0.854    0.168


LaTeX Format:
Cycle 1 & 0.231 & 0.632 & 0.121 & 0.411 & 0.658 & 0.205 \\
Cycle 2 & 0.338 & 0.600 & 0.130 & 0.389 & 0.662 & 0.229 \\
Cycle 3 & \textbf{0.391} & \textbf{0.583} & \textbf{0.093} & 0.492 & 0.854 & 0.168 \\


### **F1 STABILITY**

In [14]:
# Create pivots for both metrics
pivot_f1_max = final_epoch_df.pivot_table(
    index='config',
    columns='cycle',
    values='final_f1_max',
    aggfunc='first'
)

pivot_f1_std = final_epoch_df.pivot_table(
    index='config',
    columns='cycle',
    values='final_f1_std',
    aggfunc='first'
)

# Combine into one DataFrame with multi-level columns
combined = pd.DataFrame()

# For each cycle, add both max and std
for cycle in ['Cycle 1', 'Cycle 2', 'Cycle 3']:
    combined[f'{cycle} Max'] = pivot_f1_max[cycle]
    combined[f'{cycle} Std'] = pivot_f1_std[cycle]

# Add delta columns
combined['Total Change F1'] = pivot_f1_max['Cycle 3'] - pivot_f1_max['Cycle 1']
combined['Delta Std 1→2'] = pivot_f1_std['Cycle 2'] - pivot_f1_std['Cycle 1']
combined['Delta Std 2→3'] = pivot_f1_std['Cycle 3'] - pivot_f1_std['Cycle 2']
combined['Total Change Std'] = pivot_f1_std['Cycle 3'] - pivot_f1_std['Cycle 1']

# Sort by Cycle 3 max
combined = combined[['Cycle 1 Std', 'Cycle 2 Std', 'Cycle 3 Std', 'Cycle 1 Max', 'Cycle 2 Max', 'Cycle 3 Max', 'Delta Std 1→2', 'Delta Std 2→3', 'Total Change Std', 'Total Change F1']]
combined = combined.sort_values('Cycle 3 Max', ascending=False)

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.width', 200)

print("\n" + "=" * 120)
print("FINAL 6 EPOCHS: PEAK F1 vs CONVERGENCE STABILITY")
print("=" * 120)
print(combined.to_string())


FINAL 6 EPOCHS: PEAK F1 vs CONVERGENCE STABILITY
                     Cycle 1 Std  Cycle 2 Std  Cycle 3 Std  Cycle 1 Max  Cycle 2 Max  Cycle 3 Max  Delta Std 1→2  Delta Std 2→3  Total Change Std  Total Change F1
config                                                                                                                                                            
s1_lr5e5_bs08_wd000        0.050        0.201        0.115        0.370        0.541        0.583          0.151         -0.086             0.065            0.213
s1_lr5e5_bs08_wd010        0.148        0.118        0.078        0.529        0.538        0.583         -0.030         -0.041            -0.071            0.054
s1_lr2e5_bs16_wd001        0.017        0.181        0.086        0.308        0.600        0.545          0.164         -0.095             0.069            0.238
s1_lr5e5_bs16_wd001        0.090        0.112        0.129        0.385        0.462        0.529          0.022          0.017        

### **GAP STABILITY**

In [15]:
# Create pivots for both gap metrics
pivot_gap_std = final_epoch_df.pivot_table(
    index='config',
    columns='cycle',
    values='final_gap_std',
    aggfunc='first'
)

pivot_gap_mean = final_epoch_df.pivot_table(
    index='config',
    columns='cycle',
    values='final_gap_mean',
    aggfunc='first'
)

# Combine into one DataFrame
combined_gap = pd.DataFrame()

# For each cycle, add both max and std
for cycle in ['Cycle 1', 'Cycle 2', 'Cycle 3']:
    combined_gap[f'{cycle} Std'] = pivot_gap_std[cycle]
    combined_gap[f'{cycle} Mean'] = pivot_gap_mean[cycle]

# Add delta columns
combined_gap['Delta Std 1→2'] = pivot_gap_std['Cycle 2'] - pivot_gap_std['Cycle 1']
combined_gap['Delta Std 2→3'] = pivot_gap_std['Cycle 3'] - pivot_gap_std['Cycle 2']
combined_gap['Total Change Std'] = pivot_gap_std['Cycle 3'] - pivot_gap_std['Cycle 1']

combined_gap['Delta Gap 1→2'] = pivot_gap_mean['Cycle 2'] - pivot_gap_mean['Cycle 1']
combined_gap['Delta Gap 2→3'] = pivot_gap_mean['Cycle 3'] - pivot_gap_mean['Cycle 2']
combined_gap['Total Change Gap'] = pivot_gap_mean['Cycle 3'] - pivot_gap_mean['Cycle 1']

# Sort by Cycle 3 Max
combined_gap = combined_gap[['Cycle 1 Std', 'Cycle 2 Std', 'Cycle 3 Std', 'Cycle 1 Mean', 'Cycle 2 Mean', 'Cycle 3 Mean', 'Delta Std 1→2', 'Delta Std 2→3', 'Total Change Std', 'Delta Gap 1→2', 'Delta Gap 2→3', 'Total Change Gap']]
combined_gap = combined_gap.sort_values('Cycle 3 Mean')

pd.set_option('display.float_format', '{:.3f}'.format)

print("\n" + "=" * 120)
print("FINAL 6 EPOCHS: OVERFITTING GAP STABILITY")
print("=" * 120)

print(combined_gap.to_string())


FINAL 6 EPOCHS: OVERFITTING GAP STABILITY
                     Cycle 1 Std  Cycle 2 Std  Cycle 3 Std  Cycle 1 Mean  Cycle 2 Mean  Cycle 3 Mean  Delta Std 1→2  Delta Std 2→3  Total Change Std  Delta Gap 1→2  Delta Gap 2→3  Total Change Gap
config                                                                                                                                                                                              
s1_lr5e5_bs16_wd000        0.279        0.036        0.249         0.322         0.469         0.270         -0.244          0.213            -0.030          0.147         -0.199            -0.052
s1_lr3e5_bs16_wd000        0.308        0.278        0.258         0.309         0.261         0.317         -0.030         -0.021            -0.051         -0.048          0.056             0.008
s1_lr2e5_bs08_wd001        0.153        0.060        0.260         0.536         0.569         0.334         -0.093          0.200             0.107          0.033      